# 3. Combined 실험 - 기존 데이터셋 + ETRI 외부 데이터셋

본 노트북은 **기존 데이터셋과 ETRI 외부 데이터셋을 결합**하여 MRC 모델을 학습하고 평가합니다.

**실험 목표:**
- 데이터 증강(Data Augmentation) 효과 검증
- 기존 데이터셋 + 외부 데이터셋 결합 시 성능 변화 분석
- Baseline 및 ETRI-only 실험과의 비교
- 최적의 데이터 조합 전략 도출


## 3.1. 환경 설정 및 라이브러리 Import


In [ ]:
# 가상환경(.venv) 경로 자동 추가import sysfrom pathlib import Path# 프로젝트 루트 찾기project_root = Path().resolve().parent.parentvenv_path = project_root / ".venv"if venv_path.exists():    # Python 버전에 맞는 site-packages 경로 찾기    python_version = f"{sys.version_info.major}.{sys.version_info.minor}"    venv_site_packages = venv_path / "lib" / f"python{python_version}" / "site-packages"        # site-packages가 없으면 다른 가능한 경로 시도    if not venv_site_packages.exists():        lib_dir = venv_path / "lib"        if lib_dir.exists():            for py_dir in lib_dir.iterdir():                if py_dir.is_dir() and py_dir.name.startswith("python"):                    site_packages = py_dir / "site-packages"                    if site_packages.exists():                        venv_site_packages = site_packages                        break        if venv_site_packages.exists():        venv_path_str = str(venv_site_packages)        if venv_path_str not in sys.path:            sys.path.insert(0, venv_path_str)        print(f"✅ 가상환경(.venv) 경로 추가됨: {venv_site_packages}")    else:        print(f"⚠️ 가상환경(.venv)이 존재하지만 site-packages를 찾을 수 없습니다")else:    print(f"ℹ️ 가상환경(.venv)이 없습니다. 시스템 Python을 사용합니다")
# 노트북 독립 실행을 위한 환경 설정
import sys
from pathlib import Path

# 프로젝트 루트를 sys.path에 추가
project_root = Path().resolve().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 공통 유틸리티 import
try:
    from notebooks.utils import setup_notebook_environment, load_dataset_safely, load_json_safely
    
    # 환경 설정
    paths = setup_notebook_environment()
    print(f"✅ 프로젝트 루트: {paths['project_root']}")
    print(f"✅ 데이터 디렉토리: {paths['data_dir']}")
except ImportError as e:
    print(f"⚠️ 유틸리티 import 실패: {e}")
    paths = {
        'project_root': project_root,
        'data_dir': project_root / 'data',
        'notebook_dir': project_root / 'notebooks'
    }


# 필요한 패키지 설치
%pip install --upgrade accelerate


In [ ]:
# OpenMP 충돌 방지 설정 (Windows 환경)
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'

# 필요한 라이브러리 import
import sys
import json
import random
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from datasets import Dataset, DatasetDict, load_from_disk, concatenate_datasets
from typing import List, Dict, Tuple, Optional
from tqdm.auto import tqdm
import re
import math
import evaluate
from dataclasses import dataclass, field

# Transformers
from transformers import (
    AutoConfig,
    AutoModelForQuestionAnswering,
    AutoTokenizer,
    DataCollatorWithPadding,
    EvalPrediction,
    TrainingArguments,
    Trainer,
    set_seed,
)

# 프로젝트 루트 경로 설정
project_root = Path().resolve().parent.parent
sys.path.append(str(project_root))

# src 모듈 import
from src.config import DataTrainingArguments, ModelArguments
from src.training.trainer_qa import QuestionAnsweringTrainer
from src.utils import postprocess_qa_predictions

# 시각화 설정
plt.rcParams['figure.figsize'] = (14, 8)
sns.set_style("whitegrid")

# 재현성을 위한 시드 설정
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 디바이스 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"프로젝트 루트: {project_root}")
print(f"사용 디바이스: {device}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# 한글 폰트 설정
import matplotlib
import platform

def set_korean_font():
    system = platform.system()
    if system == 'Windows':
        font_name = 'Malgun Gothic'
    elif system == 'Darwin':  # Mac
        font_name = 'AppleGothic'
    else:  # Linux
        font_name = 'NanumGothic'
    matplotlib.rc('font', family=font_name)
    plt.rcParams['axes.unicode_minus'] = False

set_korean_font()


## 3.2. 데이터 로드


In [ ]:
# 경로 설정
data_root = project_root / "data"
train_dataset_path = data_root / "train_dataset"
etri_data_path = Path().resolve() / "data" / "etri_qa_dataset.json"

# 실험 결과 저장 경로
experiment_dir = Path().resolve() / "experiments" / "combined"
experiment_dir.mkdir(parents=True, exist_ok=True)

print(f"기존 데이터셋 경로: {train_dataset_path}")
print(f"ETRI 데이터셋 경로: {etri_data_path}")
print(f"실험 결과 저장 경로: {experiment_dir}")


In [ ]:
# 기존 데이터셋 로드
print("=== 기존 데이터셋 로드 ===")
original_datasets = load_from_disk(str(train_dataset_path))
print(f"Original Train: {len(original_datasets['train'])} samples")
print(f"Original Validation: {len(original_datasets['validation'])} samples")
print(f"컬럼: {original_datasets['train'].column_names}")


In [ ]:
# ETRI 데이터셋 로드 (00_etri_data_collection.ipynb에서 수집한 데이터)
print("\n=== ETRI 데이터셋 로드 ===")
if etri_data_path.exists():
    with open(etri_data_path, 'r', encoding='utf-8') as f:
        etri_qa_data = json.load(f)
    print(f"✅ ETRI 데이터 로드 완료: {len(etri_qa_data)} samples")
    
    # ETRI 데이터를 기존 데이터셋 형식에 맞게 변환
    # 필요한 컬럼만 추출 (confidence 제거)
    etri_data_cleaned = []
    for item in etri_qa_data:
        etri_data_cleaned.append({
            'id': item['id'],
            'title': item.get('title', ''),
            'context': item['context'],
            'question': item['question'],
            'answers': item['answers'],
        })
    
    # Train/Validation 분할 (9:1)
    random.shuffle(etri_data_cleaned)
    val_size = int(len(etri_data_cleaned) * 0.1)
    etri_train_data = etri_data_cleaned[val_size:]
    etri_val_data = etri_data_cleaned[:val_size]
    
    print(f"ETRI Train: {len(etri_train_data)} samples")
    print(f"ETRI Validation: {len(etri_val_data)} samples")
else:
    print("❌ ETRI 데이터 파일이 없습니다!")
    print(f"   경로: {etri_data_path}")
    print("\n📝 먼저 00_etri_data_collection.ipynb를 실행하여 데이터를 수집하세요.")
    etri_qa_data = None
    etri_train_data = []
    etri_val_data = []


## 3.3. 데이터셋 결합


In [ ]:
def combine_datasets(original_dataset: Dataset, external_data: List[Dict]) -> Dataset:
    """
    기존 데이터셋과 외부 데이터를 결합
    
    Args:
        original_dataset: 기존 HuggingFace Dataset
        external_data: 외부 데이터 리스트 (dict 형식)
    
    Returns:
        결합된 Dataset
    """
    # 기존 데이터셋의 컬럼 확인
    columns = original_dataset.column_names
    
    # 외부 데이터를 기존 컬럼에 맞게 정렬
    aligned_external = []
    for item in external_data:
        aligned_item = {}
        for col in columns:
            if col in item:
                aligned_item[col] = item[col]
            else:
                # 누락된 컬럼에 대한 기본값 설정
                if col == 'document_id':
                    aligned_item[col] = -1
                elif col == '__index_level_0__':
                    aligned_item[col] = -1
                else:
                    aligned_item[col] = ''
        aligned_external.append(aligned_item)
    
    # Dataset 생성 및 결합
    if aligned_external:
        external_dataset = Dataset.from_list(aligned_external)
        combined = concatenate_datasets([original_dataset, external_dataset])
        return combined
    else:
        return original_dataset


# 데이터셋 결합
if etri_train_data:
    print("=== 데이터셋 결합 ===")
    
    # Train 데이터 결합
    combined_train = combine_datasets(original_datasets['train'], etri_train_data)
    print(f"Combined Train: {len(combined_train)} samples")
    print(f"  - Original: {len(original_datasets['train'])}")
    print(f"  - ETRI: {len(etri_train_data)}")
    
    # Validation은 기존 데이터만 사용 (공정한 비교를 위해)
    combined_validation = original_datasets['validation']
    print(f"\nCombined Validation: {len(combined_validation)} samples (기존 데이터만 사용)")
    
    # DatasetDict 생성
    combined_datasets = DatasetDict({
        'train': combined_train,
        'validation': combined_validation
    })
    
    print(f"\n결합된 데이터셋 컬럼: {combined_datasets['train'].column_names}")
else:
    print("ETRI 데이터가 없어 기존 데이터셋만 사용합니다.")
    combined_datasets = original_datasets


In [ ]:
# 결합된 데이터셋 샘플 확인
print("=== 결합된 데이터셋 샘플 ===")
print("\n--- 기존 데이터 샘플 ---")
original_sample = combined_datasets['train'][0]
print(f"ID: {original_sample['id']}")
print(f"Question: {original_sample['question']}")
print(f"Answer: {original_sample['answers']['text'][0] if original_sample['answers']['text'] else 'N/A'}")

if etri_train_data:
    print("\n--- ETRI 데이터 샘플 ---")
    etri_idx = len(original_datasets['train'])  # ETRI 데이터 시작 인덱스
    if etri_idx < len(combined_datasets['train']):
        etri_sample = combined_datasets['train'][etri_idx]
        print(f"ID: {etri_sample['id']}")
        print(f"Question: {etri_sample['question']}")
        print(f"Answer: {etri_sample['answers']['text'][0] if etri_sample['answers']['text'] else 'N/A'}")


## 3.4. 모델 및 토크나이저 설정


In [ ]:
# 모델 설정
MODEL_NAME = "klue/bert-base"
MAX_SEQ_LENGTH = 384
DOC_STRIDE = 128

# 토크나이저 및 모델 로드
print(f"모델 로드 중: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model_config = AutoConfig.from_pretrained(MODEL_NAME)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME, config=model_config)

print(f"토크나이저 vocab size: {tokenizer.vocab_size}")
print(f"모델 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")


## 3.5. 데이터 전처리


In [ ]:
# 데이터 전처리 함수
def prepare_train_features(examples):
    """학습 데이터 전처리"""
    tokenized = tokenizer(
        examples['question'],
        examples['context'],
        truncation="only_second",
        max_length=MAX_SEQ_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    tokenized["start_positions"] = []
    tokenized["end_positions"] = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sequence_ids = tokenized.sequence_ids(i)
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]

        if len(answers["answer_start"]) == 0:
            tokenized["start_positions"].append(cls_index)
            tokenized["end_positions"].append(cls_index)
        else:
            start_char = answers["answer_start"][0]
            end_char = start_char + len(answers["text"][0])

            token_start_index = 0
            while sequence_ids[token_start_index] != 1:
                token_start_index += 1

            token_end_index = len(input_ids) - 1
            while sequence_ids[token_end_index] != 1:
                token_end_index -= 1

            if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
                tokenized["start_positions"].append(cls_index)
                tokenized["end_positions"].append(cls_index)
            else:
                while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                    token_start_index += 1
                tokenized["start_positions"].append(token_start_index - 1)

                while offsets[token_end_index][1] >= end_char:
                    token_end_index -= 1
                tokenized["end_positions"].append(token_end_index + 1)

    return tokenized


def prepare_validation_features(examples):
    """검증 데이터 전처리"""
    tokenized = tokenizer(
        examples['question'],
        examples['context'],
        truncation="only_second",
        max_length=MAX_SEQ_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    tokenized["example_id"] = []

    for i in range(len(tokenized["input_ids"])):
        sequence_ids = tokenized.sequence_ids(i)
        sample_index = sample_mapping[i]
        tokenized["example_id"].append(examples["id"][sample_index])
        tokenized["offset_mapping"][i] = [
            (o if sequence_ids[k] == 1 else None)
            for k, o in enumerate(tokenized["offset_mapping"][i])
        ]

    return tokenized


In [ ]:
# 데이터 전처리 수행
print("결합된 학습 데이터 전처리 중...")
train_dataset = combined_datasets['train'].map(
    prepare_train_features,
    batched=True,
    remove_columns=combined_datasets['train'].column_names
)

print("검증 데이터 전처리 중...")
validation_dataset = combined_datasets['validation'].map(
    prepare_validation_features,
    batched=True,
    remove_columns=combined_datasets['validation'].column_names
)

print(f"\n전처리된 학습 데이터 샘플 수: {len(train_dataset)}")
print(f"전처리된 검증 데이터 샘플 수: {len(validation_dataset)}")


## 3.6. 학습 설정 및 실행


In [ ]:
# 학습 인자 설정
training_args = TrainingArguments(
    output_dir=str(experiment_dir),
    do_train=True,
    do_eval=True,
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

# Data Collator
data_collator = DataCollatorWithPadding(
    tokenizer,
    pad_to_multiple_of=8 if training_args.fp16 else None
)

print("학습 설정 완료")
print(f"  - Learning Rate: {training_args.learning_rate}")
print(f"  - Batch Size: {training_args.per_device_train_batch_size}")
print(f"  - Epochs: {training_args.num_train_epochs}")
print(f"  - FP16: {training_args.fp16}")


In [ ]:
# 후처리 및 메트릭 함수
def post_processing_function(examples, features, predictions, stage="eval"):
    """예측 결과 후처리"""
    predictions = postprocess_qa_predictions(
        examples=examples,
        features=features,
        predictions=predictions,
        max_answer_length=30,
        output_dir=str(experiment_dir),
    )
    
    formatted_predictions = [
        {"id": k, "prediction_text": v} for k, v in predictions.items()
    ]
    
    if stage == "predict":
        return formatted_predictions
    
    references = [
        {"id": ex["id"], "answers": ex["answers"]}
        for ex in combined_datasets['validation']
    ]
    
    return EvalPrediction(
        predictions=formatted_predictions,
        label_ids=references
    )

metric = evaluate.load("squad")

def compute_metrics(p: EvalPrediction):
    result = metric.compute(predictions=p.predictions, references=p.label_ids)
    # Trainer가 eval_ 접두사가 붙은 메트릭을 기대하므로 접두사 추가
    return {f"eval_{k}": v for k, v in result.items()}


In [ ]:
# Trainer 초기화
trainer = QuestionAnsweringTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    eval_examples=combined_datasets['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    post_process_function=post_processing_function,
    compute_metrics=compute_metrics,
)

print("Trainer 초기화 완료")


In [ ]:
# 학습 실행
print("=" * 50)
print("Combined 데이터셋 학습 시작")
print(f"Total Train Samples: {len(combined_datasets['train'])}")
print("=" * 50)

train_result = trainer.train()

# 모델 저장
trainer.save_model()

# 학습 결과 저장
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)

print("\n학습 완료!")
print(f"Train Loss: {metrics.get('train_loss', 'N/A'):.4f}")


## 3.7. 평가


In [ ]:
# 평가 실행
print("=" * 50)
print("Combined 데이터셋 평가 시작")
print("=" * 50)

eval_metrics = trainer.evaluate()

# 평가 결과 저장
trainer.log_metrics("eval", eval_metrics)
trainer.save_metrics("eval", eval_metrics)

print("\n=== Combined 데이터셋 평가 결과 ===")
print(f"Exact Match (EM): {eval_metrics.get('eval_exact_match', 'N/A'):.2f}")
print(f"F1 Score: {eval_metrics.get('eval_f1', 'N/A'):.2f}")


In [ ]:
# 결과 요약 저장
results_summary = {
    "experiment": "combined",
    "model": MODEL_NAME,
    "original_train_samples": len(original_datasets['train']),
    "etri_train_samples": len(etri_train_data) if etri_train_data else 0,
    "total_train_samples": len(combined_datasets['train']),
    "eval_samples": len(combined_datasets['validation']),
    "epochs": training_args.num_train_epochs,
    "learning_rate": training_args.learning_rate,
    "batch_size": training_args.per_device_train_batch_size,
    "eval_exact_match": eval_metrics.get('eval_exact_match'),
    "eval_f1": eval_metrics.get('eval_f1'),
}

results_path = experiment_dir / "results_summary.json"
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(results_summary, f, ensure_ascii=False, indent=2)

print(f"\n결과 요약이 저장되었습니다: {results_path}")


## 3.8. 실험 결과 비교


In [ ]:
# 이전 실험 결과 로드 (있는 경우)
baseline_results_path = Path().resolve() / "experiments" / "baseline" / "results_summary.json"
etri_results_path = Path().resolve() / "experiments" / "etri_only" / "results_summary.json"

all_results = []

# Combined 결과
all_results.append({
    'experiment': 'Combined',
    'train_samples': len(combined_datasets['train']),
    'exact_match': eval_metrics.get('eval_exact_match', 0),
    'f1': eval_metrics.get('eval_f1', 0)
})

# Baseline 결과 (있는 경우)
if baseline_results_path.exists():
    with open(baseline_results_path, 'r', encoding='utf-8') as f:
        baseline_results = json.load(f)
    all_results.append({
        'experiment': 'Baseline',
        'train_samples': baseline_results.get('train_samples', 0),
        'exact_match': baseline_results.get('eval_exact_match', 0),
        'f1': baseline_results.get('eval_f1', 0)
    })
    print("✓ Baseline 결과 로드 완료")
else:
    print("⚠️ Baseline 결과 파일이 없습니다.")

# ETRI-only 결과 (있는 경우)
if etri_results_path.exists():
    with open(etri_results_path, 'r', encoding='utf-8') as f:
        etri_results = json.load(f)
    all_results.append({
        'experiment': 'ETRI Only',
        'train_samples': etri_results.get('train_samples', 0),
        'exact_match': etri_results.get('eval_exact_match', 0),
        'f1': etri_results.get('eval_f1', 0)
    })
    print("✓ ETRI-only 결과 로드 완료")
else:
    print("⚠️ ETRI-only 결과 파일이 없습니다.")

# 결과 DataFrame 생성
results_df = pd.DataFrame(all_results)
print("\n=== 실험 결과 비교 ===")
print(results_df.to_string(index=False))


## 3.9. 결과 시각화


In [ ]:
# 결과 시각화
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 색상 팔레트
colors = {'Combined': '#27ae60', 'Baseline': '#3498db', 'ETRI Only': '#e67e22'}

# 1. Exact Match 비교
if len(results_df) > 0:
    exp_names = results_df['experiment'].tolist()
    em_scores = results_df['exact_match'].tolist()
    bar_colors = [colors.get(exp, '#95a5a6') for exp in exp_names]
    
    bars1 = axes[0].bar(exp_names, em_scores, color=bar_colors)
    axes[0].set_ylabel('Score')
    axes[0].set_title('Exact Match 비교')
    axes[0].set_ylim(0, 100)
    
    for bar, val in zip(bars1, em_scores):
        if val > 0:
            axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                         f'{val:.2f}', ha='center', va='bottom', fontsize=11)

# 2. F1 Score 비교
if len(results_df) > 0:
    f1_scores = results_df['f1'].tolist()
    
    bars2 = axes[1].bar(exp_names, f1_scores, color=bar_colors)
    axes[1].set_ylabel('Score')
    axes[1].set_title('F1 Score 비교')
    axes[1].set_ylim(0, 100)
    
    for bar, val in zip(bars2, f1_scores):
        if val > 0:
            axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                         f'{val:.2f}', ha='center', va='bottom', fontsize=11)

# 3. 데이터셋 크기 비교
dataset_labels = ['Original', 'ETRI']
dataset_sizes = [
    len(original_datasets['train']),
    len(etri_train_data) if etri_train_data else 0
]

axes[2].pie(dataset_sizes, labels=dataset_labels, autopct='%1.1f%%',
            colors=['#3498db', '#e67e22'], startangle=90)
axes[2].set_title(f'Combined 데이터셋 구성\n(Total: {sum(dataset_sizes)})')

plt.tight_layout()
plt.savefig(experiment_dir / "combined_results.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\n그래프 저장 완료: {experiment_dir / 'combined_results.png'}")


## 3.10. 결론 및 인사이트


In [ ]:
# 실험 결론 출력
print("=" * 60)
print("실험 결론")
print("=" * 60)

print(f"\n📊 Combined 실험 결과:")
print(f"   - Total Train Samples: {len(combined_datasets['train'])}")
print(f"   - Exact Match: {eval_metrics.get('eval_exact_match', 0):.2f}")
print(f"   - F1 Score: {eval_metrics.get('eval_f1', 0):.2f}")

# 성능 비교 (Baseline 결과가 있는 경우)
if baseline_results_path.exists():
    baseline_em = baseline_results.get('eval_exact_match', 0)
    baseline_f1 = baseline_results.get('eval_f1', 0)
    combined_em = eval_metrics.get('eval_exact_match', 0)
    combined_f1 = eval_metrics.get('eval_f1', 0)
    
    em_diff = combined_em - baseline_em
    f1_diff = combined_f1 - baseline_f1
    
    print(f"\n📈 Baseline 대비 성능 변화:")
    print(f"   - Exact Match: {em_diff:+.2f} ({'개선' if em_diff > 0 else '하락' if em_diff < 0 else '동일'})")
    print(f"   - F1 Score: {f1_diff:+.2f} ({'개선' if f1_diff > 0 else '하락' if f1_diff < 0 else '동일'})")
    
    if em_diff > 0 and f1_diff > 0:
        print("\n✅ 결론: 외부 데이터셋 추가가 성능 향상에 기여했습니다!")
    elif em_diff < 0 or f1_diff < 0:
        print("\n⚠️ 결론: 외부 데이터셋 품질 검토가 필요합니다.")
    else:
        print("\n📌 결론: 외부 데이터셋 추가 효과가 미미합니다.")

print("\n" + "=" * 60)
print("다음 단계 권장사항:")
print("=" * 60)
print("1. 외부 데이터 품질 검토 및 필터링")
print("2. 데이터 비율 조정 실험 (Original:ETRI = 1:1, 2:1, etc.)")
print("3. Hard Negative 샘플링과 결합한 실험")
print("4. 다양한 모델 아키텍처에서의 검증")
